In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import urllib.parse

from sklearn.metrics.pairwise import cosine_similarity

### Loading Animelist

In [2]:
df_AnimeList = pd.DataFrame()
for f in list(Path('../Scraping/Season').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_AnimeList = pd.concat([df_AnimeList, df_Read])

del df_Read

In [3]:
df_AnimeList['MAL_id'] = df_AnimeList['Link'].apply(lambda x : int(re.findall(r'anime/(\d+)',x)[0]))
df_AnimeList = df_AnimeList.drop(columns=['Link', 'Unnamed: 0'])
df_AnimeList = df_AnimeList.set_index('MAL_id')
df_AnimeList = df_AnimeList.drop_duplicates()
df_AnimeList

,Title
MAL_id,
30,Shinseiki Evangelion
852,Gokinjo Monogatari
495,Bakuretsu Hunters
1567,Kaitou Saint Tail
116,Shinpi no Sekai El-Hazard (TV)
...,...
54145,Cardfight!! Vanguard: D Series Season 9
53722,Bullet/Bullet
60726,My Melody & Kuromi


### Loading User Ratings

In [4]:
df_Raw = pd.DataFrame()
for f in list(Path('../Scraping/Users/Watchlist').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_Read['User'] = f.name.replace('.csv', '')
    df_Raw = pd.concat([df_Raw, df_Read])

del df_Read

In [5]:
df_Raw = df_Raw.drop(columns=['Unnamed: 0'])
df_Raw = df_Raw.dropna()

df_Raw['MAL_id'] = df_Raw['MAL_id'].astype(int)
df_Raw['Title'] = df_Raw['Title'].apply(lambda x : urllib.parse.unquote(x).replace('_', ' '))
df_Raw

,MAL_id,Title,Score,Type,User
4,40028,Shingeki no Kyojin The Final Season,10.0,TV,-Norza-
20,40938,Hige wo Soru Soshite Joshikousei wo Hirou,9.0,TV,-Norza-
32,1735,Naruto Shippuuden,9.0,TV,-Norza-
59,3455,To LOVE-Ru,7.0,TV,-Norza-
61,16011,Tokyo Ravens,8.0,TV,-Norza-
...,...,...,...,...,...
224,49987,Papa Katsu,6.0,OVA,_AverageWatcher_
226,59120,Shounen ga Otona ni Natta Natsu,3.0,OVA,_AverageWatcher_
227,49329,Soshite Watashi wa Sensei ni,4.0,OVA,_AverageWatcher_
228,55524,Tsundero Series,4.0,OVA,_AverageWatcher_


### User-Anime Score Matrix

In [6]:
matrix_Raw = df_Raw.pivot_table(index='User', columns='MAL_id', values='Score', aggfunc='mean')
matrix_Raw = matrix_Raw.fillna(0)
matrix_Raw = matrix_Raw.astype(np.int8)
matrix_Raw

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
User,,,,,,,,,,,,,,,,,,,,,
-Norza-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
-Xythus-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0_Kira_0,0,0,0,0,0,0,9,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0neiros,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0rekiHoutarou,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zenjamibu,0,0,10,0,0,0,0,0,10,0,...,0,0,0,0,0,0,0,0,0,0
zhorahh,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
zikangue,7,0,0,0,0,9,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### User-User Similarity Matrix

In [7]:
matrix_Users = pd.DataFrame(cosine_similarity(matrix_Raw), index=matrix_Raw.index, columns=matrix_Raw.index)
matrix_Users.head(10)

User,-Norza-,-Xythus-,0_Kira_0,0neiros,0rekiHoutarou,1fallenleaf,23Jcelle,24xnyk,2alvin,7Gungnir7,...,yoghurtstick,yuki-chi,yumenade,zairiko,zazanya,zenjamibu,zhorahh,zikangue,zilch16,zwrei
User,,,,,,,,,,,,,,,,,,,,,
-Norza-,1.000000,0.248460,0.173178,0.162018,0.221289,0.092652,0.025384,0.069213,0.00000,0.243878,...,0.000000,0.059677,0.000000,0.100901,0.156391,0.097872,0.000000,0.254551,0.036748,0.082479
-Xythus-,0.248460,1.000000,0.437163,0.313810,0.476413,0.327651,0.099013,0.047527,0.00000,0.258417,...,0.000000,0.270025,0.038106,0.261281,0.240923,0.070281,0.137909,0.354235,0.106830,0.041473
0_Kira_0,0.173178,0.437163,1.000000,0.275815,0.488422,0.153598,0.085671,0.013791,0.00000,0.137117,...,0.065829,0.290977,0.126029,0.295504,0.118000,0.067108,0.086132,0.204728,0.152791,0.027679
0neiros,0.162018,0.313810,0.275815,1.000000,0.338728,0.156473,0.056910,0.072180,0.00000,0.221010,...,0.064854,0.132939,0.021297,0.156020,0.096997,0.109626,0.097975,0.201366,0.090689,0.065143
0rekiHoutarou,0.221289,0.476413,0.488422,0.338728,1.000000,0.168382,0.101346,0.016921,0.00000,0.270766,...,0.053845,0.178602,0.074367,0.247761,0.145101,0.126125,0.121759,0.302219,0.134844,0.000000
1fallenleaf,0.092652,0.327651,0.153598,0.156473,0.168382,1.000000,0.157927,0.000000,0.00000,0.106113,...,0.000000,0.193860,0.051606,0.114186,0.174275,0.020706,0.155515,0.262846,0.066640,0.155109
23Jcelle,0.025384,0.099013,0.085671,0.056910,0.101346,0.157927,1.000000,0.000000,0.00000,0.063952,...,0.000000,0.220161,0.239149,0.080268,0.095261,0.081112,0.054260,0.060072,0.209521,0.100262
24xnyk,0.069213,0.047527,0.013791,0.072180,0.016921,0.000000,0.000000,1.000000,0.09202,0.072606,...,0.000000,0.000000,0.000000,0.015199,0.000000,0.008214,0.000000,0.030624,0.011235,0.000000
2alvin,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.092020,1.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Test User Actual_Hana
Gather Top 10 Most Similar Users

In [8]:
USER = 'Actual_Hana'

similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:11]
similar_users

User
xiuzuki          0.385509
MysticMira       0.374602
kannequi         0.339587
xkourokox        0.337709
0rekiHoutarou    0.331107
0_Kira_0         0.323373
arrei            0.321667
HyMaster         0.318735
usernamezxc      0.318273
lucy_kawaii      0.315495
Name: Actual_Hana, dtype: float64

## Shows User Watched. Others have Watched Some

In [9]:
alreadyWatched = matrix_Raw.loc[USER] > 0
matrix_Raw.loc[np.append(USER, similar_users.index), alreadyWatched]

MAL_id,20,22,121,123,226,270,355,356,384,853,...,22145,22199,22319,22535,23273,23283,23317,23673,27899,28223
User,,,,,,,,,,,,,,,,,,,,,
Actual_Hana,10,9,10,10,9,9,9,9,10,9,...,8,9,10,10,10,10,8,10,10,10
xiuzuki,10,0,6,0,0,0,10,0,0,8,...,0,8,8,0,8,0,0,0,9,0
MysticMira,4,0,0,0,0,0,0,0,0,6,...,8,7,7,8,8,0,0,0,9,0
kannequi,8,0,0,0,0,0,0,0,0,0,...,0,0,5,0,0,0,0,0,5,7
xkourokox,0,10,0,0,0,0,0,0,0,10,...,0,8,9,0,8,7,0,7,8,0
0rekiHoutarou,0,0,0,5,0,0,5,0,0,0,...,0,8,7,8,8,4,0,7,3,0
0_Kira_0,0,0,0,0,7,0,8,8,0,0,...,0,0,9,9,9,9,0,0,9,8
arrei,0,0,0,0,0,0,0,0,0,9,...,0,0,8,9,8,8,0,0,7,10
HyMaster,0,0,0,0,0,0,0,10,0,0,...,0,8,9,9,8,0,0,0,9,8


## Shows User hasn't Watched. Others have

In [10]:
matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]
matrix_Other_User_Ratings

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
User,,,,,,,,,,,,,,,,,,,,,
xiuzuki,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
MysticMira,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
kannequi,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xkourokox,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0rekiHoutarou,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0_Kira_0,0,0,0,0,0,0,9,0,0,0,...,0,0,0,0,0,0,0,0,0,0
arrei,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
HyMaster,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
usernamezxc,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Recommendations' Avg Score by Similar Users

In [11]:
reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:10]
df_Reccomendations = df_AnimeList.loc[reccomended_Shows.index].copy()
df_Reccomendations['Avg_Score'] = reccomended_Shows.values
df_Reccomendations

,Title,Avg_Score
MAL_id,,
16894,Kuroko no Basket 2nd Season,6.8
11111,Another,6.1
4181,Clannad: After Story,5.9
2904,Code Geass: Hangyaku no Lelouch R2,5.8
7054,Kaichou wa Maid-sama!,5.6
28171,Shokugeki no Souma,5.5
6347,Baka to Test to Shoukanjuu,5.3
14813,Yahari Ore no Seishun Love Comedy wa Machigatt...,5.3
12189,Hyouka,5.2


In [12]:
def GetPersonalTop(USER, n = 10):
    USER_top = matrix_Raw.loc[USER].sort_values(ascending=False)[:n]
    
    if USER_top.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[USER_top.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[USER_top.index].drop_duplicates()
    
    df['Personal_Score'] = USER_top.values
    df = df.rename(columns={'Title': f"{USER}'s Top {n}"})
    return df

    
def Recommend(USER, n = 10):
    similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:n+1]
    alreadyWatched = matrix_Raw.loc[USER] > 0
    matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]

    reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:n]

    if reccomended_Shows.index.isin(df_AnimeList.index).all():
        df = df_AnimeList.loc[reccomended_Shows.index].copy()
    else:
        df = df_Raw[['MAL_id', 'Title']].copy().set_index('MAL_id').loc[reccomended_Shows.index].drop_duplicates()
        
    df['Avg_Recommenders_Score'] = reccomended_Shows.values
    df = df.rename(columns={'Title': f"Others' Top {n}"})
        
    return df

## Test Mainstream Viewer

In [13]:
USER = '-Norza-'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,-Norza-'s Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,36286,Re:Zero kara Hajimeru Isekai Seikatsu - Memory...,10,38691,Dr. Stone,8.2
1,30276,One Punch Man,10,17895,Golden Time,7.3
2,32380,Kono Subarashii Sekai ni Shukufuku wo!: Kono S...,10,40748,Jujutsu Kaisen,7.3
3,33674,No Game No Life: Zero,10,37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,7.0
4,13759,Sakura-sou no Pet na Kanojo,10,41389,Tonikaku Kawaii,6.8
5,32281,Kimi no Na wa.,10,15809,Hataraku Maou-sama!,6.7
6,40591,Kaguya-sama wa Kokurasetai? Tensai-tachi no Re...,10,11061,Hunter x Hunter (2011),6.7
7,37105,Grand Blue,10,48736,Sono Bisque Doll wa Koi wo Suru,6.7
8,22961,Date A Live II: Kurumi Star Festival,10,28171,Shokugeki no Souma,6.6
9,38524,Shingeki no Kyojin Season 3 Part 2,10,9253,Steins;Gate,6.6


## Test Watched Only One Punch Man (Cold Start)

In [14]:
USER = 'EXUPUROOOSHUN'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,EXUPUROOOSHUN's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,30276,One Punch Man,10,16498,Shingeki no Kyojin,3.7
1,35631,Mahouka x Mameshiba,0,22319,Tokyo Ghoul,2.9
2,35695,Keito no Yousei Knit to Wool,0,21,One Piece,2.9
3,35694,Kirin the Noop,0,32281,Kimi no Na wa.,2.9
4,35686,Doraemon 2005 Specials,0,28121,Dungeon ni Deai wo Motomeru no wa Machigatteir...,2.6
5,35678,Hibike Euphonium Movie 3 Chikai no Finale,0,2236,Toki wo Kakeru Shoujo,2.0
6,35677,Liz to Aoi Tori,0,23755,Nanatsu no Taizai,2.0
7,35676,Berserk Majo no Tsuisou,0,28851,Koe no Katachi,1.9
8,35673,Yama no Susume Omoide Present,0,38000,Kimetsu no Yaiba,1.9
9,35672,Yama no Susume Third Season,0,1604,Katekyou Hitman Reborn!,1.9


## Test Max Ranker : Everything 10

In [15]:
USER = 'Muhclis'
pd.concat([GetPersonalTop(USER,100).reset_index(), Recommend(USER,100).reset_index()], axis=1)

,MAL_id,Muhclis's Top 100,Personal_Score,MAL_id,Others' Top 100,Avg_Recommenders_Score
0,36564,Kamiusagi Rope x Boruto Naruto Next Generations,10,31240,Re:Zero kara Hajimeru Isekai Seikatsu,5.40
1,40520,Chii-chan Kaihatsu Nikki,10,31964,Boku no Hero Academia,5.09
2,9849,Akina to Onsen de H Shiyo,10,38691,Dr. Stone,4.94
3,50346,Yofukashi no Uta,10,33486,Boku no Hero Academia 2nd Season,4.75
4,3944,Dokidoki Oyako Lesson Oshiete H na Obenkyou,10,36456,Boku no Hero Academia 3rd Season,4.20
...,...,...,...,...,...,...
95,15227,Kono Sekai no Katasumi ni,10,25537,Fate/stay night Movie: Heaven's Feel - I. Pres...,2.11
96,35968,Wotaku ni Koi wa Muzukashii,10,18153,Kyoukai no Kanata,2.10
97,10075,Naruto x UT,10,35839,Sora yori mo Tooi Basho,2.10
98,40855,Hajimete no Hitozuma,10,41353,The God of High School,2.10


## AOT Fan

In [16]:
USER = 'Math082r'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,Math082r's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,2251,Baccano!,10,9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,6.6
1,28851,Koe no Katachi,10,2904,Code Geass: Hangyaku no Lelouch R2,6.1
2,3297,Aria the Origination,9,28171,Shokugeki no Souma,6.0
3,59571,Shingeki no Kyojin Movie: Kanketsu-hen - The L...,9,1575,Code Geass: Hangyaku no Lelouch,6.0
4,4181,Clannad: After Story,9,5114,Fullmetal Alchemist: Brotherhood,5.8
5,51535,Shingeki no Kyojin: The Final Season - Kankets...,9,21,One Piece,5.7
6,48583,Shingeki no Kyojin: The Final Season Part 2,9,14345,Btooom!,5.6
7,40028,Shingeki no Kyojin: The Final Season,9,35788,Shokugeki no Souma: San no Sara,5.3
8,48553,Akebi-chan no Sailor-fuku,8,51019,Kimetsu no Yaiba: Katanakaji no Sato-hen,5.3
9,8676,Amagami SS,8,38000,Kimetsu no Yaiba,5.2


## Drama/Seinnen (Monogatari / Garden of Sinners)

In [17]:
USER = 'Shizawari'
pd.concat([GetPersonalTop(USER, 15).reset_index(), Recommend(USER, 15).reset_index()], axis=1)

,MAL_id,Shizawari's Top 15,Personal_Score,MAL_id,Others' Top 15,Avg_Recommenders_Score
0,56784,Bleach Sennen Kessen-hen - Soukoku-tan,10,32281,Kimi no Na wa.,9.533333
1,33352,Violet Evergarden,10,23273,Shigatsu wa Kimi no Uso,7.266667
2,31758,Kizumonogatari III Reiketsu-hen,10,38691,Dr. Stone,7.266667
3,4282,Kara no Kyoukai Movie 5 Mujun Rasen,10,28999,Charlotte,7.200000
4,31757,Kizumonogatari II Nekketsu-hen,10,5114,Fullmetal Alchemist: Brotherhood,7.000000
5,48653,Yoru ni Kakeru,10,9253,Steins;Gate,7.000000
6,5081,Bakemonogatari,10,11061,Hunter x Hunter (2011),6.800000
7,47917,Bocchi the Rock,10,45576,Mushoku Tensei: Isekai Ittara Honki Dasu Part 2,6.733333
8,11597,Nisemonogatari,10,1535,Death Note,6.600000
9,37521,Vinland Saga,10,42897,Horimiya,6.333333


## Gooner

In [18]:
USER = 'omanko1'
pd.concat([GetPersonalTop(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,omanko1's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,50192,Akane wa Tsumare Somerareru,10,11827,Kuroinu: Kedakaki Seijo wa Hakudaku ni Somaru,7.3
1,59119,Imaria,10,38470,Saimin Seishidou,6.7
2,49295,Chizuru-chan Kaihatsu Nikki,10,24641,Baku Ane: Otouto Shibocchau zo! The Animation,6.3
3,55850,Tsuma ni Damatte Sokubaikai ni Ikun ja Nakatta,9,36720,Youkoso! Sukebe Elf no Mori e,6.3
4,22429,Itadaki! Seieki♥,9,39042,Gaki ni Modotte Yarinaoshi!!!,6.0
5,39530,Ecchi na Oneechan ni Shiboraretai,9,48698,Mako-chan Kaihatsu Nikki,5.6
6,23479,Itadaki! Seieki♥ Special,9,33322,JK Bitch ni Shiboraretai,5.5
7,48468,Uchi no Otouto Maji de Dekain Dakedo Mi ni Konai?,9,38866,Seikatsu Shuukan The Animation,5.4
8,40337,Aibeya The Animation,9,21829,Fella Pure: Mitarashi-san Chi no Jijou The Ani...,5.4
9,2539,Itazura The Animation,9,29575,Mankitsu Happening,5.4


# Review
Compared to content based, this method feels more like a black box approach.

It relies on the shrared ratings between users that will often be difficult to visually interpret considering the matrix's size. More than that, most of what is recommended is Mainstream due to majority ruling.

The only most notable distinction we could make was that a Porn viewer is correctly recommendeded explicit content from other users. 

# Item-Item Approach

In [19]:
matrix_Anime = pd.DataFrame(cosine_similarity(matrix_Raw.T), index=matrix_Raw.columns, columns=matrix_Raw.columns)
matrix_Anime.head()

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
MAL_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.523740,0.251923,0.198553,0.152584,0.270273,0.215722,0.147494,0.174425,0.351816,...,0.0,0.048286,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.127753
5,0.523740,1.000000,0.249615,0.221811,0.067172,0.143359,0.144907,0.161095,0.099056,0.185684,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.220498
6,0.251923,0.249615,1.000000,0.304063,0.181470,0.303731,0.128665,0.215105,0.225270,0.139862,...,0.0,0.069875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
7,0.198553,0.221811,0.304063,1.000000,0.110322,0.167338,0.196158,0.319702,0.248818,0.134699,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
8,0.152584,0.067172,0.181470,0.110322,1.000000,0.264351,0.071500,0.210201,0.138004,0.060939,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


## Test Harem : Rent a Girlfirend

Expecting Subsequent Seasons + Other harem Anime 

In [20]:
def ReccommendByItem(ANIME_ID : int, n = 10):
    similar_ID = matrix_Anime.loc[ANIME_ID].sort_values(ascending=False)[:n+1]
    return pd.concat([df_Raw.set_index('MAL_id').loc[similar_ID.index]['Title'].drop_duplicates(), similar_ID], axis=1)

In [21]:
ReccommendByItem(40839)

,Title,40839
MAL_id,,
40839,Kanojo Okarishimasu,1.000000
42963,Kanojo Okarishimasu 2nd Season,0.612562
38101,5-toubun no Hanayome,0.579389
39783,5-toubun no Hanayome ∬,0.576021
40938,Hige wo Soru Soshite Joshikousei wo Hirou,0.572088
37450,Seishun Buta Yarou wa Bunny Girl Senpai no Yum...,0.570653
39547,Yahari Ore no Seishun Love Comedy wa Machigatt...,0.568032
35849,Darling in the FranXX,0.564820
38000,Kimetsu no Yaiba,0.538332


## Test Niche Gambling : Kakegurui
Expecting shows related to competition.

In [22]:
ReccommendByItem(34933)

,Title,34933
MAL_id,,
34933,Kakegurui,1.000000
37086,Kakegurui××,0.756991
38000,Kimetsu no Yaiba,0.559406
35849,Darling in the FranXX,0.554964
37999,Kaguya-sama wa Kokurasetai Tensai-tachi no Re...,0.552416
32282,Shokugeki no Souma Ni no Sara,0.551986
28851,Koe no Katachi,0.551112
35507,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu e,0.549069
40591,Kaguya-sama wa Kokurasetai Tensai-tachi no Ren...,0.525967
